# User Data Usage Analytics Pipeline

Collects usage analytics across **Tableau, Metabase, Trino (Data Lake), and Vertica**.

### Two modes
| Mode | Set in Cell 1 | What it does |
|------|--------------|-------------|
| **`users`** | `TARGET_UIDS` list | Find top tables & dashboards for given users |
| **`tables`** | `TARGET_TABLES` list | Find who queries given tables + SQL patterns (search by table_name part) |

Typical flow: run **users** mode → get popular tables → run **tables** mode on those tables to collect SQL patterns.

**Modules:**
1. **Tableau** — users mode: dashboard views by user; tables mode: dashboards whose datasources reference the table
2. **Metabase** — query log from `v_query_log`, extract SQL from `query` JSON, parse referenced tables
3. **Data Lake (Trino)** — table usage from analytical query logs
4. **Vertica** — table usage from `query_requests_history`

**Output:** Unified CSV with top tables, dashboards, sample queries, and user coverage stats.

In [ ]:
# ================================================================
# Cell 1 — Config & Inputs
# ================================================================

# ---- Pipeline mode: 'users' or 'tables' ----
PIPELINE_MODE = 'users'  # <-- change to 'tables' for table-based search

# ---- Mode 'users': target user UIDs ----
TARGET_UIDS = [
    'a.starostina',
    'mark.moiseev',
]

# ---- Mode 'tables': target table names (just the table_name part, not full path) ----
TARGET_TABLES = [
    'dat_affiliate_appsflyer_installs_rpt',
    'sse_client_dm_out',
]

# ---- Date range ----
FROM_DT = '2025-01-01'
TO_DT   = '2026-02-01'

# ---- Output CSV paths ----
OUTPUT_TABLEAU_CSV  = 'user_usage_tableau.csv'
OUTPUT_METABASE_CSV = 'user_usage_metabase.csv'
OUTPUT_TRINO_CSV    = 'user_usage_trino.csv'
OUTPUT_VERTICA_CSV  = 'user_usage_vertica.csv'
OUTPUT_UNIFIED_CSV  = 'user_usage_unified.csv'

# ---- Shared constants ----
EXCLUDE_SCHEMAS = {
    'trading_work', 'tech_ml_platform_work', 'partnership_work', 'marketing_work',
    'finance_work', 'protection_work', 'trading_pricing_work', 'information_schema',
    'system', 'common_uploads', 'commercial_work', 'tech_data_platform_work',
    'payments_work', 'relationship_work', 'tech_data_platform_dbt_tmp_mart',
}

TOP_K_JOINS   = 15
TOP_K_COLS    = 20
TOP_K_QUERIES = 5
MAX_QUERY_LEN = 2500
CHUNKSIZE     = 250_000

# ---- SQL helpers ----
TARGET_UIDS_SQL = '(' + ', '.join(f"'{u}'" for u in TARGET_UIDS) + ')'
TARGET_TABLES_LOWER = [t.lower() for t in TARGET_TABLES]

# SQL LIKE conditions for table name search (matches table_name anywhere in query text)
TARGET_TABLES_LIKE_SQL = ' OR '.join(
    f"lower(query) LIKE '%{t}%'" for t in TARGET_TABLES_LOWER
)

print(f'Pipeline mode: {PIPELINE_MODE}')
if PIPELINE_MODE == 'users':
    print(f'Target UIDs:   {TARGET_UIDS}')
else:
    print(f'Target tables: {TARGET_TABLES}')
print(f'Date range:    {FROM_DT} → {TO_DT}')

In [ ]:
# ================================================================
# Cell 2 — Trino Connections
# ================================================================

import trino
from trino.dbapi import connect
from trino.auth import BasicAuthentication
from sqlalchemy.engine import create_engine
import urllib3
from urllib3.exceptions import InsecureRequestWarning
import pandas as pd
import re

urllib3.disable_warnings(InsecureRequestWarning)

# --- Option A: use credscram (non-interactive) ---
try:
    import credscram
    _ldap_user = credscram.creds.ldap_name
    _ldap_pass = credscram.creds.ldap_pass
    print(f'[auth] Using credscram as {_ldap_user}')
except ImportError:
    from getpass import getpass
    _ldap_user = input('LDAP username: ').strip()
    _ldap_pass = getpass('LDAP password: ')
    print(f'[auth] Using manual credentials as {_ldap_user}')

# 1) SQLAlchemy engine for %sql magic
trino_db = create_engine(
    'trino://trino.exness.io:443',
    connect_args={
        'auth': trino.auth.BasicAuthentication(_ldap_user, _ldap_pass),
        'http_scheme': 'https',
        'verify': '/etc/ssl/certs/ca-certificates.crt',
    })

%load_ext sql
%sql trino_db
%config SqlMagic.displaylimit = 100

# 2) Raw dbapi connection for streaming pandas reads
conn = connect(
    host='trino.exness.io',
    port=443,
    auth=BasicAuthentication(_ldap_user, _ldap_pass),
    http_scheme='https',
    verify=False,
)

print('[ok] Trino connections ready')

In [ ]:
# ================================================================
# Cell 3 — User Directory Lookup (Jira object store)
# In 'users' mode: lookup TARGET_UIDS
# In 'tables' mode: still needed for enrichment — loads all users (no filter)
# ================================================================

import json

UID_RE = re.compile(r"^\s*['\"]?(?P<uid>[^'\",@\s]+(?:\.[^'\",@\s]+)?)['\"]?\s*$")

def normalize_uid(val: str) -> str:
    """Strip quotes/whitespace, take left of '@', force lower; keep name.surname only."""
    if val is None:
        return ''
    s = str(val).strip()
    if '@' in s:
        s = s.split('@', 1)[0]
    m = UID_RE.match(s)
    if not m:
        return s.strip(" '\"").lower()
    return m.group('uid').lower()


# In 'users' mode filter to TARGET_UIDS; in 'tables' mode load full directory for enrichment
_uid_filter = f"WHERE uid IN {TARGET_UIDS_SQL}" if PIPELINE_MODE == 'users' else ''

sql_users = f"""
SELECT uid, job, direction, department, group_job, uid_manager
FROM (
    SELECT
        SPLIT_PART(ep.label, '@', 1) AS uid,
        MAX_BY(val_text, val_id) FILTER (WHERE ota_id IN (463, 5736))  AS user_name,
        MAX_BY(val_text, val_id) FILTER (WHERE ota_id IN (458, 5727))  AS job,
        MAX_BY(val_text, val_id) FILTER (WHERE ota_id = 456)           AS direction,
        MAX_BY(val_text, val_id) FILTER (WHERE ota_id IN (457, 5728))  AS department,
        MAX_BY(val_text, val_id) FILTER (WHERE ota_id = 849)           AS group_job,
        MAX_BY(regexp_extract(val_text, '^uid=([^,]+)', 1), val_id)
            FILTER (WHERE ota_id IN (5730, 32287))                     AS uid_manager,
        MAX_BY(boolean_value, val_id) FILTER (WHERE ota_id = 5753)     AS deactivated
    FROM (
        SELECT DISTINCT
            SPLIT_PART(ep.label, '@', 1)          AS uid,
            attr.object_type_attribute_id          AS ota_id,
            c.id                                   AS val_id,
            c.text_value                           AS val_text,
            c.boolean_value
        FROM delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj ep
        LEFT JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr attr
            ON attr.object_id = ep.id
        LEFT JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr_val c
            ON c.object_attribute_id = attr.id
        WHERE regexp_like(trim(lower(SPLIT_PART(ep.label, '@', 1))), '^[a-z]+[.][a-z]+$')
          AND ep.label NOT LIKE '%/%'
          AND attr.object_type_attribute_id IN (463, 5736, 458, 456, 457, 849, 5730, 5753, 5727, 5728, 32287)
    ) src_jira
    GROUP BY 1
)
{_uid_filter}
"""

cursor = conn.cursor()
cursor.execute(sql_users)
users_df = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])
users_df = users_df.astype('string')
users_df['uid'] = users_df['uid'].map(normalize_uid)

print(f'[ok] users_df: {len(users_df)} rows')
display(users_df.head(10))

In [ ]:
# ================================================================
# Cell 4 — Module 1: Tableau Usage
# 'users' mode: filter by user UIDs (who viewed which dashboards)
# 'tables' mode: filter by datasource name containing TARGET_TABLES
# ================================================================

# Build the WHERE clause depending on mode
if PIPELINE_MODE == 'users':
    _tableau_filter = f"AND users.uid IN {TARGET_UIDS_SQL}"
else:
    # In tables mode: match datasource name containing any of the target table names
    _ds_conditions = ' OR '.join(
        f"lower(ds.workbook_name) LIKE '%{t}%' OR EXISTS (SELECT 1 FROM UNNEST(ds.ds_name) AS x(n) WHERE lower(n) LIKE '%{t}%')"
        for t in TARGET_TABLES_LOWER
    )
    _tableau_filter = f"AND ({_ds_conditions})"

sql_tableau = f"""
WITH uid_job_info AS (
    SELECT uid, job, direction, department, group_job, uid_manager
    FROM (
        SELECT
            SPLIT_PART(ep.label, '@', 1) AS uid,
            max(CASE WHEN object_type_attribute_id = 463 THEN c.text_value END) AS user_name,
            max(CASE WHEN object_type_attribute_id = 458 THEN c.text_value END) AS job,
            max(CASE WHEN object_type_attribute_id = 456 THEN c.text_value END) AS direction,
            max(CASE WHEN object_type_attribute_id = 457 THEN c.text_value END) AS department,
            max(CASE WHEN object_type_attribute_id = 849 THEN c.text_value END) AS group_job,
            max(CASE WHEN object_type_attribute_id IN (5730)
                     THEN regexp_extract(c.text_value, '^uid=([^,]+)', 1) END) AS uid_manager
        FROM delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj ep
        LEFT JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr attr
            ON attr.object_id = ep.id
        LEFT JOIN delta.tech_data_platform_core.jira_ao_8542f1_ifj_obj_attr_val c
            ON c.object_attribute_id = attr.id
        WHERE regexp_like(trim(lower(SPLIT_PART(ep.label, '@', 1))), '^[a-z]+[.][a-z]+$')
          AND ep.label NOT LIKE '%/%'
        GROUP BY 1
    ) WHERE job IS NOT NULL
),

wb_datasources AS (
    SELECT
        wb.id AS workbook_id,
        wb.name AS workbook_name,
        wb.description AS workbook_description,
        array_agg(DISTINCT ds.name ORDER BY ds.name) AS ds_name,
        array_agg(DISTINCT ds.db_class ORDER BY ds.db_class) AS db_class
    FROM delta.tech_data_platform_core.tableau_workbooks wb
    LEFT JOIN delta.tech_data_platform_core.tableau_datasources ds
        ON ds.parent_workbook_id = wb.id
    LEFT JOIN delta.tech_data_platform_core.tableau_hist_datasources hds
        ON hds.id = ds.id
    WHERE ds.reduced_data_id IS NULL
      AND lower(ds.name) NOT LIKE '%dummy%'
      AND lower(ds.name) NOT LIKE '%placeholder%'
      AND lower(ds.name) NOT LIKE '%sandbox%'
      AND lower(ds.name) NOT LIKE '%working_days%'
    GROUP BY 1, 2, 3
)

SELECT
    hv.name AS view_name,
    hv.repository_url AS view_url,
    p.name AS project_name,
    ds.ds_name,
    ds.db_class,
    ds.workbook_name,
    hu.name AS user_uid,
    users.department,
    users.direction,
    date_trunc('month', he.created_at) AS event_month,
    count(DISTINCT he.id) AS event_count
FROM delta.tech_data_platform_core.tableau_historical_events he
JOIN delta.tech_data_platform_core.tableau_historical_event_types et
    ON et.type_id = he.historical_event_type_id
   AND et.action_type NOT IN ('Delete', 'Create')
JOIN delta.tech_data_platform_core.tableau_hist_projects p
    ON p.id = he.hist_project_id
LEFT JOIN delta.tech_data_platform_core.tableau_hist_views hv
    ON he.hist_view_id = hv.id
LEFT JOIN delta.tech_data_platform_core.tableau_views v
    ON hv.view_id = v.id
LEFT JOIN delta.tech_data_platform_core.tableau_hist_workbooks hwb
    ON he.hist_workbook_id = hwb.id
LEFT JOIN wb_datasources ds
    ON ds.workbook_id = hwb.workbook_id
LEFT JOIN delta.tech_data_platform_core.tableau_workbooks wb
    ON hwb.workbook_id = wb.id
LEFT JOIN delta.tech_data_platform_core.tableau_hist_users hu
    ON hu.id = he.hist_actor_user_id
LEFT JOIN uid_job_info users
    ON users.uid = hu.name
WHERE he.created_at >= date'{FROM_DT}'
  AND he.created_at <  date'{TO_DT}'
  AND et.name IN ('Access View', 'Send E-Mail')
  {_tableau_filter}
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
"""

cursor = conn.cursor()
cursor.execute(sql_tableau)
tableau_df = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])

tableau_df.to_csv(OUTPUT_TABLEAU_CSV, index=False)
print(f'[ok] Tableau usage ({PIPELINE_MODE} mode): {len(tableau_df)} rows → {OUTPUT_TABLEAU_CSV}')
display(tableau_df.head(10))

In [ ]:
# ================================================================
# Cell 5 — Module 2: Metabase Usage
# Source: pg_data_quality__ba_db_metabase3_ro.public.v_query_log
# ================================================================

METABASE_QUERY_LOG = 'pg_data_quality__ba_db_metabase3_ro.public.v_query_log'

# v_query_log columns:
#   entity_id, started_at, running_time_seconds, result_rows, is_native,
#   query_source, error, user_id, card_id, card_qualified_id,
#   dashboard_id, dashboard_qualified_id, pulse_id, database_id,
#   database_qualified_id, cache_hit, action_id, action_qualified_id, query
#
# `query` is a JSON blob. For native queries the SQL lives in:
#   json_extract_scalar(query, '$.stages[0].native')   -- Trino JSON path
# For MBQL queries it's structured JSON (source-table / source-card references).

# --- Build WHERE depending on mode ---
if PIPELINE_MODE == 'users':
    # First: resolve TARGET_UIDS → Metabase user_ids via core_user
    _mb_user_cond = ' OR '.join(f"lower(cu.email) LIKE '{uid}@%'" for uid in TARGET_UIDS)
    sql_mb = f"""
    WITH mb_users AS (
        SELECT cu.id AS user_id, cu.email, cu.first_name, cu.last_name
        FROM pg_data_quality__ba_db_metabase3_ro.public.core_user cu
        WHERE {_mb_user_cond}
    )
    SELECT
        vl.entity_id,
        vl.started_at,
        vl.running_time_seconds,
        vl.result_rows,
        vl.is_native,
        vl.query_source,
        vl.user_id,
        mu.email AS user_email,
        vl.card_id,
        vl.card_qualified_id,
        vl.dashboard_id,
        vl.dashboard_qualified_id,
        vl.database_id,
        vl.cache_hit,
        vl.query
    FROM {METABASE_QUERY_LOG} vl
    JOIN mb_users mu ON mu.user_id = vl.user_id
    WHERE vl.started_at >= timestamp '{FROM_DT} 00:00:00'
      AND vl.started_at <  timestamp '{TO_DT} 00:00:00'
    """
else:
    # In tables mode: search for table name in the query JSON text
    _mb_table_cond = ' OR '.join(
        f"lower(vl.query) LIKE '%{t}%'" for t in TARGET_TABLES_LOWER
    )
    sql_mb = f"""
    SELECT
        vl.entity_id,
        vl.started_at,
        vl.running_time_seconds,
        vl.result_rows,
        vl.is_native,
        vl.query_source,
        vl.user_id,
        CAST(NULL AS varchar) AS user_email,
        vl.card_id,
        vl.card_qualified_id,
        vl.dashboard_id,
        vl.dashboard_qualified_id,
        vl.database_id,
        vl.cache_hit,
        vl.query
    FROM {METABASE_QUERY_LOG} vl
    WHERE vl.started_at >= timestamp '{FROM_DT} 00:00:00'
      AND vl.started_at <  timestamp '{TO_DT} 00:00:00'
      AND ({_mb_table_cond})
    """

cursor = conn.cursor()
cursor.execute(sql_mb)
mb_raw = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])
print(f'[metabase] Fetched {len(mb_raw)} query log rows')

# --- Extract native SQL from query JSON ---
def extract_metabase_sql(query_json_str):
    """Extract SQL from Metabase v_query_log `query` JSON field.
    Native queries: stages[0].native
    MBQL queries: return None (structured, no raw SQL)
    """
    if not query_json_str or not isinstance(query_json_str, str):
        return None
    try:
        dq = json.loads(query_json_str)
    except (json.JSONDecodeError, TypeError):
        return None
    # Try stages[0].native (new format seen in v_query_log)
    stages = dq.get('stages', [])
    if stages and isinstance(stages, list):
        native = stages[0].get('native') if isinstance(stages[0], dict) else None
        if native and isinstance(native, str):
            return native
    # Fallback: old format native.query
    native_block = dq.get('native', {})
    if isinstance(native_block, dict):
        q = native_block.get('query')
        if q:
            return q
    return None

def extract_metabase_source_table(query_json_str):
    """For MBQL queries: extract source-table or source-card reference."""
    if not query_json_str or not isinstance(query_json_str, str):
        return None
    try:
        dq = json.loads(query_json_str)
    except (json.JSONDecodeError, TypeError):
        return None
    q = dq.get('query', {})
    if isinstance(q, dict):
        st = q.get('source-table')
        if st is not None:
            return f'metabase_table_id:{st}'
        sc = q.get('source-card')
        if sc is not None:
            return f'metabase_card:{sc}'
    return None

mb_raw['extracted_sql'] = mb_raw['query'].apply(extract_metabase_sql)
mb_raw['source_ref'] = mb_raw['query'].apply(extract_metabase_source_table)

# Parse referenced tables from native SQL (uses extract_tables from Cell 6)
mb_raw['referenced_tables'] = mb_raw['extracted_sql'].apply(
    lambda sql: sorted(extract_tables(sql)) if sql else []
)

# --- Aggregate per card/dashboard ---
mb_raw['has_sql'] = mb_raw['extracted_sql'].notna()

metabase_agg = (
    mb_raw
    .groupby(['card_id', 'card_qualified_id', 'dashboard_id', 'dashboard_qualified_id'])
    .agg(
        exec_count=('entity_id', 'count'),
        unique_users=('user_id', 'nunique'),
        users_list=('user_id', lambda s: sorted(set(str(x) for x in s.dropna()))),
        avg_running_time=('running_time_seconds', 'mean'),
        total_result_rows=('result_rows', 'sum'),
        cache_hit_pct=('cache_hit', 'mean'),
        last_used=('started_at', 'max'),
        sample_sql=('extracted_sql', lambda s: next((x for x in s if x), None)),
        referenced_tables=('referenced_tables', lambda s: sorted(set(
            t for tables in s for t in tables
        ))),
        source_ref=('source_ref', 'first'),
        query_source=('query_source', 'first'),
    )
    .reset_index()
)

metabase_agg['referenced_tables'] = metabase_agg['referenced_tables'].apply(json.dumps)
metabase_agg['sample_sql'] = metabase_agg['sample_sql'].apply(
    lambda s: s[:MAX_QUERY_LEN] if s else None
)

metabase_out = metabase_agg
metabase_out.to_csv(OUTPUT_METABASE_CSV, index=False)
print(f'[ok] Metabase usage ({PIPELINE_MODE} mode): {len(metabase_out)} rows → {OUTPUT_METABASE_CSV}')
display(metabase_out.head(10))

In [ ]:
# ================================================================
# Cell 6 — SQL Parsing Utilities
# Reuse: datalake_catalog_draft.ipynb cell 13 / top_dashboards.ipynb cell 28
# ================================================================

import re
import hashlib
from collections import Counter, defaultdict

# ---------------- REGEX & HELPERS ----------------

def _strip_quotes(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = s.strip().replace('"', '').replace('`', '')
    return re.sub(r'^\[|\]$', '', s)

def _norm_ident(s: str) -> str:
    s = _strip_quotes(s)
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return re.sub(r'[^a-z0-9._]', '', s)

CTE_NAME_RE     = re.compile(r'(?is)\b([`"\[]?[A-Za-z_][A-Za-z0-9_]*[`"\]]?)\s+as\s*\(')
TABLE_CLAUSE_RE = re.compile(r'(?is)\b(from|join|into|update|merge\s+into|table)\s+([`"\[\]A-Za-z0-9_.]+)')
JOIN_TARGET_RE  = re.compile(r'(?is)\bjoin\s+([`"\[\]A-Za-z0-9_.]+)')
COMMENT_RE      = re.compile(r'(--[^\n]*\n)|(/\*.*?\*/)', re.S | re.I)

SKIP_QUERY_RE = re.compile(
    r'''(?is)^\s*(?:--[^\n]*\n\s*|/\*.*?\*/\s*)*
        (?:create\s+(?:or\s+replace\s+)?(?:table|view)\b
           |insert\s+(?:overwrite\s+)?into\b)''',
    re.VERBOSE
)

EXCLUDE_NAMES    = {'unnest', 'lateral', 'values', 'if', 'case', 'true', 'false', 'null'}
EXCLUDE_PREFIXES = ('information_schema.', 'system.', 'sys.', 'sqlite_', '__temp__', '_tmp_', 'tmp_')

# ---------------- SQL PARSING FUNCTIONS ----------------

def _is_ddl_dml_to_skip(sql: str) -> bool:
    return bool(sql and SKIP_QUERY_RE.search(sql))

def extract_cte_names(sql: str) -> set:
    if not isinstance(sql, str) or not sql.strip():
        return set()
    if not re.search(r'(?is)^\s*with\b', sql):
        return set()
    return {_norm_ident(m.group(1)) for m in CTE_NAME_RE.finditer(sql)}

def looks_like_table(name: str) -> bool:
    if '.' not in name:
        return False
    if any(name.startswith(p) for p in EXCLUDE_PREFIXES):
        return False
    if name in EXCLUDE_NAMES:
        return False
    return True

def extract_tables(sql: str) -> set:
    ctes = extract_cte_names(sql)
    found = set()
    for m in TABLE_CLAUSE_RE.finditer(sql or ''):
        ident = _norm_ident(m.group(2))
        if ident and ident not in ctes and looks_like_table(ident):
            found.add(ident)
    return found

def extract_join_targets(sql: str) -> set:
    out = set()
    for m in JOIN_TARGET_RE.finditer(sql or ''):
        ident = _norm_ident(m.group(1))
        if ident and looks_like_table(ident):
            out.add(ident)
    return out

def _to_base_table(ident: str) -> str | None:
    parts = ident.split('.')
    if len(parts) == 3:
        return f'{parts[1]}.{parts[2]}'
    if len(parts) == 2:
        return ident
    return None

def _schema_of_base(bt: str) -> str | None:
    return bt.split('.', 1)[0] if '.' in bt else None

def find_columns_for_table(sql: str, base_table: str) -> list:
    try:
        schema, tbl = base_table.split('.', 1)
    except ValueError:
        return []
    return re.findall(
        rf'(?i)\b(?:{schema}\s*\.\s*)?{tbl}\s*\.\s*([a-z_][a-z0-9_]*)\b',
        sql or ''
    )

def _strip_sql_comments(sql: str) -> str:
    return COMMENT_RE.sub(' ', sql or '')

def _is_trivial_for_base(sql: str, base_table: str) -> bool:
    s = _strip_sql_comments(sql).lower()
    if re.search(r'\b(where|join|group\s+by|having|union|window)\b', s):
        return False
    if re.search(r'\b(count|sum|avg|min|max)\s*\(|\bover\b', s):
        return False
    tables = {_to_base_table(t) for t in extract_tables(sql) if _to_base_table(t)}
    return tables == {base_table}

def _normalize_and_truncate_query(q: str) -> str:
    q = re.sub(r'\s+', ' ', q or '').strip()
    return q[:MAX_QUERY_LEN]

def hash64(s: str) -> bytes:
    return hashlib.blake2b(s.encode('utf-8'), digest_size=8).digest()

def query_mentions_tables(sql: str, table_names: list) -> bool:
    """Check if SQL mentions any of the target table names (case-insensitive, by table_name part)."""
    if not sql:
        return False
    sql_lower = sql.lower()
    return any(t in sql_lower for t in table_names)

print('[ok] SQL parsing utilities loaded')

In [ ]:
# ================================================================
# Cell 7 — Module 3: Data Lake (Trino) Table Usage
# 'users' mode: filter by user UIDs
# 'tables' mode: filter by table name in query text (LIKE)
# ================================================================

TRINO_INPUT_TABLE = 'delta.tech_data_platform_work.astarostina_trino_analytical_queries'

EXCLUDE_JOIN_SCHEMAS = {
    'trading_work', 'tech_ml_platform_work', 'trading_pricing_work', 'tech_data_platform_work',
    'partnership_work', 'marketing_work', 'finance_work', 'protection_work', 'commercial_work',
    'payments_work', 'common_uploads', 'jdbc', 'tech_data_platform_dbt_tmp_mart',
}

# Build WHERE depending on mode
if PIPELINE_MODE == 'users':
    _trino_where = f"WHERE u IN {TARGET_UIDS_SQL} AND request_type != 'app'"
else:
    _trino_table_cond = ' OR '.join(
        f"lower(query_norm) LIKE '%{t}%'" for t in TARGET_TABLES_LOWER
    )
    _trino_where = f"WHERE request_type != 'app' AND ({_trino_table_cond})"

# ---------------- ACCUMULATORS ----------------

trino_join_counter     = defaultdict(Counter)
trino_join_kw_counter  = defaultdict(Counter)
trino_column_counter   = defaultdict(Counter)
trino_sample_counter   = defaultdict(Counter)

trino_total_weight     = defaultdict(int)
trino_users_set        = defaultdict(set)
trino_query_hash_set   = defaultdict(set)
trino_catalogs_set     = defaultdict(set)

# ---------------- TRINO STREAM ----------------

SQL_TRINO = f"""
SELECT
    query_norm AS query,
    u AS "user",
    count(*) AS num_queries
FROM {TRINO_INPUT_TABLE}
{_trino_where}
GROUP BY 1, 2
"""

df_iter = pd.read_sql_query(SQL_TRINO, conn, chunksize=CHUNKSIZE)

# ---------------- MAIN LOOP ----------------

total_rows = 0

for chunk in df_iter:
    chunk = chunk.dropna(subset=['query']).drop_duplicates(subset=['query', 'user'])

    for q, u, w in zip(chunk['query'], chunk['user'], chunk['num_queries']):
        total_rows += 1
        w = int(w) if w and w > 0 else 1

        if _is_ddl_dml_to_skip(q):
            continue

        tables = extract_tables(q)
        joins  = extract_join_targets(q)
        if not tables:
            continue

        base_set = {_to_base_table(t) for t in tables if _to_base_table(t)}
        base_set = {b for b in base_set if _schema_of_base(b) not in EXCLUDE_JOIN_SCHEMAS}
        if not base_set:
            continue

        # In tables mode: only keep rows that actually reference target tables
        if PIPELINE_MODE == 'tables':
            if not any(t in bt for bt in base_set for t in TARGET_TABLES_LOWER):
                continue

        for t in tables:
            p = _norm_ident(t).split('.')
            if len(p) == 3 and p[0] in {'delta', 'iceberg'}:
                trino_catalogs_set[f'{p[1]}.{p[2]}'].add(p[0])

        q_short = _normalize_and_truncate_query(q)

        for bt in base_set:
            trino_total_weight[bt] += w
            if u:
                trino_users_set[bt].add(u)

            for o in base_set - {bt}:
                trino_join_counter[bt][o] += w

            for o in {_to_base_table(j) for j in joins if _to_base_table(j)} - {bt}:
                trino_join_kw_counter[bt][o] += w

            if not _is_trivial_for_base(q, bt):
                for c in find_columns_for_table(q, bt):
                    trino_column_counter[bt][c] += w
                trino_sample_counter[bt][q_short] += w
                trino_query_hash_set[bt].add(hash64(q_short))

    if total_rows % 500_000 == 0:
        print(f'[progress] processed ~{total_rows:,} rows')

# ---------------- BUILD OUTPUT ----------------

def top_list(cnt: Counter, k: int):
    return [x for x, _ in cnt.most_common(k)]

trino_rows = []
for bt in sorted(set(trino_users_set)):
    trino_rows.append({
        'source': 'trino',
        'base_table': bt,
        'queries_cnt': len(trino_query_hash_set[bt]),
        'queries_cnt_w': trino_total_weight[bt],
        'users_cnt': len(trino_users_set[bt]),
        'users_list': sorted(trino_users_set[bt]),
        'catalogs': sorted(trino_catalogs_set.get(bt, [])),
        'top_joins': top_list(trino_join_kw_counter[bt] or trino_join_counter[bt], TOP_K_JOINS),
        'top_columns': top_list(trino_column_counter[bt], TOP_K_COLS),
        'sample_queries': top_list(trino_sample_counter[bt], TOP_K_QUERIES),
    })

trino_out = pd.DataFrame(trino_rows)
trino_out.to_csv(OUTPUT_TRINO_CSV, index=False)

print(f'[done] Trino table stats ({PIPELINE_MODE} mode): {len(trino_out)} tables | processed={total_rows:,} → {OUTPUT_TRINO_CSV}')

In [ ]:
# ================================================================
# Cell 8 — Module 4: Vertica Table Usage
# 'users' mode: filter by user_name
# 'tables' mode: filter by table name in query text (LIKE)
# ================================================================

VERTICA_SOURCE = 'vertica_data_quality__ro.prometheus.query_requests_history'

# Build WHERE depending on mode
if PIPELINE_MODE == 'users':
    _vertica_filter = f"AND user_name IN {TARGET_UIDS_SQL}"
else:
    _vertica_table_cond = ' OR '.join(
        f"lower(request) LIKE '%{t}%'" for t in TARGET_TABLES_LOWER
    )
    _vertica_filter = f"AND ({_vertica_table_cond})"

SQL_VERTICA = f"""
SELECT
    request AS query,
    user_name AS "user",
    start_timestamp
FROM {VERTICA_SOURCE}
WHERE request_type = 'QUERY'
  AND start_timestamp >= date'{FROM_DT}'
  AND start_timestamp <  date'{TO_DT}'
  AND user_name NOT IN ('etlmeta')
  AND request != 'select now()::timestamptz'
  {_vertica_filter}
"""

cursor = conn.cursor()
cursor.execute(SQL_VERTICA)
vertica_raw = pd.DataFrame(cursor.fetchall(), columns=[c[0] for c in cursor.description])

print(f'[vertica] Fetched {len(vertica_raw)} query rows ({PIPELINE_MODE} mode)')

# ---------------- ACCUMULATORS (same structure as Module 3) ----------------

v_join_counter     = defaultdict(Counter)
v_join_kw_counter  = defaultdict(Counter)
v_column_counter   = defaultdict(Counter)
v_sample_counter   = defaultdict(Counter)

v_total_weight     = defaultdict(int)
v_users_set        = defaultdict(set)
v_query_hash_set   = defaultdict(set)

# ---------------- PARSE LOOP ----------------

for q, u in zip(vertica_raw['query'], vertica_raw['user']):
    if not q or not isinstance(q, str):
        continue
    if _is_ddl_dml_to_skip(q):
        continue

    tables = extract_tables(q)
    joins  = extract_join_targets(q)
    if not tables:
        continue

    base_set = {_to_base_table(t) for t in tables if _to_base_table(t)}
    base_set = {b for b in base_set if _schema_of_base(b) not in EXCLUDE_JOIN_SCHEMAS}
    if not base_set:
        continue

    # In tables mode: only keep rows that actually reference target tables
    if PIPELINE_MODE == 'tables':
        if not any(t in bt for bt in base_set for t in TARGET_TABLES_LOWER):
            continue

    q_short = _normalize_and_truncate_query(q)

    for bt in base_set:
        v_total_weight[bt] += 1
        if u:
            v_users_set[bt].add(u)

        for o in base_set - {bt}:
            v_join_counter[bt][o] += 1

        for o in {_to_base_table(j) for j in joins if _to_base_table(j)} - {bt}:
            v_join_kw_counter[bt][o] += 1

        if not _is_trivial_for_base(q, bt):
            for c in find_columns_for_table(q, bt):
                v_column_counter[bt][c] += 1
            v_sample_counter[bt][q_short] += 1
            v_query_hash_set[bt].add(hash64(q_short))

# ---------------- BUILD OUTPUT ----------------

vertica_rows = []
for bt in sorted(set(v_users_set)):
    vertica_rows.append({
        'source': 'vertica',
        'base_table': bt,
        'queries_cnt': len(v_query_hash_set[bt]),
        'queries_cnt_w': v_total_weight[bt],
        'users_cnt': len(v_users_set[bt]),
        'users_list': sorted(v_users_set[bt]),
        'catalogs': [],
        'top_joins': top_list(v_join_kw_counter[bt] or v_join_counter[bt], TOP_K_JOINS),
        'top_columns': top_list(v_column_counter[bt], TOP_K_COLS),
        'sample_queries': top_list(v_sample_counter[bt], TOP_K_QUERIES),
    })

vertica_out = pd.DataFrame(vertica_rows)
vertica_out.to_csv(OUTPUT_VERTICA_CSV, index=False)

print(f'[done] Vertica table stats ({PIPELINE_MODE} mode): {len(vertica_out)} tables → {OUTPUT_VERTICA_CSV}')

In [ ]:
# ================================================================
# Cell 9 — Unified Output & Summary Statistics
# ================================================================

import json

# Helper: enrich user list with org info from users_df
def _enrich_users_org(uid_list):
    uids = uid_list if isinstance(uid_list, list) else []
    matched = users_df[users_df['uid'].isin(uids)]
    return (
        sorted(set(matched['department'].dropna())),
        sorted(set(matched['direction'].dropna())),
    )

# ---- 1) Tableau → unified schema ----
if len(tableau_df) > 0:
    tableau_unified = (
        tableau_df
        .groupby(['view_name', 'view_url', 'project_name'])
        .agg(
            query_count=('event_count', 'sum'),
            unique_users=('user_uid', 'nunique'),
            users_list=('user_uid', lambda s: sorted(set(s.dropna()))),
            departments=('department', lambda s: sorted(set(s.dropna()))),
            directions=('direction', lambda s: sorted(set(s.dropna()))),
            ds_names=('ds_name', lambda s: sorted(set(
                d for arr in s.dropna() for d in (arr if isinstance(arr, list) else [str(arr)])
                if d and str(d) != 'None'
            ))),
            last_used=('event_month', 'max'),
        )
        .reset_index()
    )
    tableau_unified['source'] = 'tableau'
    tableau_unified['asset_name'] = tableau_unified['project_name'] + ' / ' + tableau_unified['view_name']
    tableau_unified['asset_type'] = 'dashboard'
    tableau_unified['referenced_tables'] = tableau_unified['ds_names'].apply(json.dumps)
    tableau_unified['sample_queries'] = '[]'
else:
    tableau_unified = pd.DataFrame()

# ---- 2) Metabase → unified schema ----
if len(metabase_out) > 0:
    metabase_unified = metabase_out.copy()
    metabase_unified['source'] = 'metabase'
    metabase_unified['asset_name'] = metabase_unified.apply(
        lambda r: (
            f"dashboard:{r['dashboard_qualified_id']} / card:{r['card_qualified_id']}"
            if pd.notna(r.get('dashboard_qualified_id'))
            else f"card:{r['card_qualified_id']}"
        ),
        axis=1
    )
    metabase_unified['asset_type'] = 'question'
    metabase_unified['query_count'] = metabase_unified['exec_count']
    metabase_unified['sample_queries'] = metabase_unified['sample_sql'].apply(
        lambda s: json.dumps([s[:MAX_QUERY_LEN]]) if s else '[]'
    )

    # Enrich with user metadata
    if PIPELINE_MODE == 'users':
        user_departments = users_df['department'].dropna().unique().tolist()
        user_directions  = users_df['direction'].dropna().unique().tolist()
        metabase_unified['departments'] = json.dumps(sorted(set(user_departments)))
        metabase_unified['directions']  = json.dumps(sorted(set(user_directions)))
    else:
        metabase_unified['departments'] = '[]'
        metabase_unified['directions']  = '[]'
    metabase_unified['users_list'] = metabase_unified['users_list'].apply(
        lambda v: json.dumps(v) if isinstance(v, list) else v
    )
else:
    metabase_unified = pd.DataFrame()

# ---- 3) Trino → unified schema ----
if len(trino_out) > 0:
    trino_unified = trino_out.copy()
    trino_unified['source'] = 'trino'
    trino_unified['asset_name'] = trino_unified['base_table']
    trino_unified['asset_type'] = 'table'
    trino_unified['query_count'] = trino_unified['queries_cnt_w']
    trino_unified['unique_users'] = trino_unified['users_cnt']
    trino_unified['referenced_tables'] = trino_unified['base_table'].apply(lambda t: json.dumps([t]))
    trino_unified['last_used'] = None
    trino_unified[['departments', 'directions']] = trino_unified['users_list'].apply(
        lambda u: pd.Series(_enrich_users_org(u))
    )
    trino_unified['departments'] = trino_unified['departments'].apply(json.dumps)
    trino_unified['directions']  = trino_unified['directions'].apply(json.dumps)
else:
    trino_unified = pd.DataFrame()

# ---- 4) Vertica → unified schema ----
if len(vertica_out) > 0:
    vertica_unified = vertica_out.copy()
    vertica_unified['source'] = 'vertica'
    vertica_unified['asset_name'] = vertica_unified['base_table']
    vertica_unified['asset_type'] = 'table'
    vertica_unified['query_count'] = vertica_unified['queries_cnt_w']
    vertica_unified['unique_users'] = vertica_unified['users_cnt']
    vertica_unified['referenced_tables'] = vertica_unified['base_table'].apply(lambda t: json.dumps([t]))
    vertica_unified['last_used'] = None
    vertica_unified[['departments', 'directions']] = vertica_unified['users_list'].apply(
        lambda u: pd.Series(_enrich_users_org(u))
    )
    vertica_unified['departments'] = vertica_unified['departments'].apply(json.dumps)
    vertica_unified['directions']  = vertica_unified['directions'].apply(json.dumps)
else:
    vertica_unified = pd.DataFrame()

# ---- 5) Union all ----
UNIFIED_COLS = [
    'source', 'asset_name', 'asset_type', 'referenced_tables',
    'query_count', 'unique_users', 'users_list',
    'departments', 'directions',
    'sample_queries', 'last_used',
]

parts = []
for df_part in [tableau_unified, metabase_unified, trino_unified, vertica_unified]:
    if len(df_part) == 0:
        continue
    for col in UNIFIED_COLS:
        if col not in df_part.columns:
            df_part[col] = None
    # Serialize list columns to JSON strings for CSV compatibility
    for col in ['users_list', 'sample_queries']:
        if col in df_part.columns:
            df_part[col] = df_part[col].apply(
                lambda v: json.dumps(v) if isinstance(v, list) else v
            )
    parts.append(df_part[UNIFIED_COLS])

unified = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=UNIFIED_COLS)

unified.to_csv(OUTPUT_UNIFIED_CSV, index=False)
print(f'[done] Unified output: {len(unified)} rows → {OUTPUT_UNIFIED_CSV}')

# ---- 6) Summary Statistics ----
print(f'\nPipeline mode: {PIPELINE_MODE}')
print('='*60)
print('SUMMARY')
print('='*60)

if len(unified) > 0:
    print(f'\nRows per source:')
    print(unified['source'].value_counts().to_string())

    print(f'\nRows per asset_type:')
    print(unified['asset_type'].value_counts().to_string())

    # Top 20 tables across all sources
    tables_only = unified[unified['asset_type'] == 'table'].copy()
    if not tables_only.empty:
        tables_only['query_count'] = pd.to_numeric(tables_only['query_count'], errors='coerce')
        print(f'\n--- Top 20 tables by query_count ---')
        display(
            tables_only
            .sort_values('query_count', ascending=False)
            .head(20)[['source', 'asset_name', 'query_count', 'unique_users', 'departments', 'directions']]
        )

    # Top 10 dashboards/questions
    dash_only = unified[unified['asset_type'].isin(['dashboard', 'question'])].copy()
    if not dash_only.empty:
        dash_only['query_count'] = pd.to_numeric(dash_only['query_count'], errors='coerce')
        print(f'\n--- Top 10 dashboards/questions by query_count ---')
        display(
            dash_only
            .sort_values('query_count', ascending=False)
            .head(10)[['source', 'asset_name', 'query_count', 'unique_users']]
        )

# Coverage stats
print(f'\n--- Coverage ---')
if PIPELINE_MODE == 'users':
    print(f'Target UIDs:         {len(TARGET_UIDS)}')
    print(f'Users in directory:  {len(users_df)}')
    print(f'Users in Tableau:    {tableau_df["user_uid"].nunique() if len(tableau_df) else 0}')
    print(f'Users in Metabase:   {mb_raw["user_id"].nunique() if len(mb_raw) else 0}')
else:
    print(f'Target tables:       {TARGET_TABLES}')
    print(f'Tableau dashboards:  {len(tableau_df["view_name"].unique()) if len(tableau_df) else 0}')
    print(f'Metabase cards:      {len(metabase_out)}')
print(f'Trino tables found:  {len(trino_out)}')
print(f'Vertica tables found:{len(vertica_out)}')